# Inferencia Estadística TP4

**Eduardo Nicolas Sanchez Lopez**

Costo de oportunidad del inventario de bolsas de fibra de 600 gramos. Una celda de preparación, una por cada uno de los cinco puntos y una para la actividad extra. Cada respuesta reúne resultado, método, justificación y límites. Ejecutar de arriba hacia abajo.

Los importes monetarios se muestran con dos decimales; los cálculos conservan toda su precisión. No se generan archivos PDF al ejecutar este notebook.

In [1]:
import sys
from pathlib import Path

PROJECT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src/analysis.py").is_file()
)
sys.path.insert(0, str(PROJECT / "src"))
print("Entorno local preparado.")

import math
from IPython.display import HTML, display
from analysis import analyze, opportunity_cost, value_inventory
from notebook_view import render_answer

Entorno local preparado.


In [2]:
# Punto 1. Costo de adquisición: subtotal informado, sin margen de venta.
result = analyze(PROJECT)
unit_cost_ars = result["pallet_cost_ars"] / result["pallet_bags"]
assert math.isclose(unit_cost_ars, result["unit_cost_ars"])
display(HTML(render_answer(PROJECT, 1, result)))

Componente informado,Celda,Importe por palet (ARS)
Precio de fábrica (palet de 1.260 bolsas),E59,"5.490.190,00"
"Impuesto a las ganancias, según planilla",E60,"1.811.764,00"
"Impuesto a los ingresos brutos, según planilla",E61,"164.705,00"
Descarga,E62,"100.000,00"
Seguro,E63,"7.566,00"
Costo puesto en Mendoza,E65,"7.574.225,00"


In [3]:
# Punto 2. Conversión a USD con una cotización fechada común.
unit_cost_usd = unit_cost_ars / result["exchange_rate"]["ars_per_usd"]
assert math.isclose(unit_cost_usd, result["unit_cost_usd"])
display(HTML(render_answer(PROJECT, 2, result)))

In [4]:
# Punto 3. Media de estados corregidos, no media ponderada por días.
corrected_balances = [row["corrected_balance"] for row in result["stock_audit"]]
mean_stock = sum(corrected_balances) / len(corrected_balances)
stock_value_usd = value_inventory(mean_stock, unit_cost_ars, result["exchange_rate"]["ars_per_usd"])
assert math.isclose(stock_value_usd, result["stock_value_usd"])
display(HTML(render_answer(PROJECT, 3, result)))

Verificación de la reconstrucción,Bolsas
Stock inicial + compras − ventas,3.396 + 47.880 − 47.528
Stock final corregido,3.748
Promedio de los 37 estados de inventario,"4.112,51"


In [5]:
# Punto 4. Retornos totales NAV: misma moneda, fecha y horizonte anual.
annual_returns = {fund["ticker"]: fund["annual_return"] for fund in result["funds"]}
display(HTML(render_answer(PROJECT, 4, result)))

Fondo,Período y base comunes,Retorno anual
AOR,Un año al 31/05/2025 · NAV · USD,"9,901322 % [1]"
SGOV,Un año al 31/05/2025 · NAV · USD,"4,780024 % [2]"


In [6]:
# Punto 5. Escenarios de costo de oportunidad, no prueba de superioridad.
annual_costs = {ticker: opportunity_cost(stock_value_usd, rate) for ticker, rate in annual_returns.items()}
for fund in result["funds"]:
    assert math.isclose(annual_costs[fund["ticker"]], fund["annual_cost_usd"])
display(HTML(render_answer(PROJECT, 5, result)))

Alternativa,Capital (USD),Tasa anual,Costo (USD/año)
AOR,"20.883,53","9,901322 %","2.067,75"
SGOV,"20.883,53","4,780024 %","998,24"


In [7]:
# Actividad extra. Regla predictiva simbólica: no se inventa un stock óptimo.
# La cobertura, la ventana mensual y la reposición requieren validación.
display(HTML(render_answer(PROJECT, 6, result)))

## Fuentes y datos

Planilla original: `data/raw/inventory.xlsx`. Se conserva el criterio de reconstrucción del TP2 y se excluye el saldo repetido de la fila 52.

Las tasas y el MEP son referencias históricas fechadas, no cotizaciones actuales. Consulta de fuentes: 21/09/2026.

- [1] [iShares: AOR, retorno total NAV a un año al 31/05/2025](https://www.ishares.com/varnish-api/blk-one01-product-data/product-data/api/v2/get-product-data?appSubType=ISHARES&appType=PRODUCT_PAGE&component=performance.returns.average&locale=en_US&portfolioId=239756&targetSite=us-ishares&userType=individual&excludeContent=true&asOfDate=20250531&includeConfig=true).
- [2] [iShares: SGOV, retorno total NAV a un año al 31/05/2025](https://www.ishares.com/varnish-api/blk-one01-product-data/product-data/api/v2/get-product-data?appSubType=ISHARES&appType=PRODUCT_PAGE&component=performance.returns.average&locale=en_US&portfolioId=314116&targetSite=us-ishares&userType=individual&excludeContent=true&asOfDate=20250531&includeConfig=true).
- [3] [La Nación: cotización MEP de cierre, 02/06/2025](https://www.lanacion.com.ar/economia/dolar/dolar-hoy-tras-tocar-los-1200-las-cotizaciones-vuelven-a-operar-a-la-baja-nid02062025).
- [4] [iShares: SGOV, descripción, riesgo y metodología](https://www.ishares.com/us/products/314116/ishares-0-3-month-treasury-bond-etf).
- [5] [iShares: AOR, descripción y metodología](https://www.ishares.com/us/products/239756/ishares-growth-allocation-etf).
- [6] [iShares: informe anual AOR, identidad del fondo](https://www.ishares.com/us/literature/annual-report/ar-aor-en.pdf).
- [7] [Universidad Siglo 21: Módulo 3, lectura 4, Pruebas de hipótesis para diferencia de medias](https://meca.ues21.edu.ar/canvas/0GRADO2A2023/inferenciaestadistica/L12/M3L4.pdf).
- [8] [Universidad Siglo 21: Módulo 4, lectura 1, Ajuste de curvas, regresión y correlación](https://meca.ues21.edu.ar/canvas/0GRADO2A2023/inferenciaestadistica/L13/M4L1.pdf).
- [9] [Gershwin, S. B. (2016). Inventory. MIT OpenCourseWare, 2.854: modelo EOQ, pp. 43–50](https://ocw.mit.edu/courses/2-854-introduction-to-manufacturing-systems-fall-2016/6c8fec8f99bcf7059b73b82e96d43901_MIT2_854F16_Inventory.pdf).
- [10] [NIST. Dataplot Reference Manual: Prediction Limits (mean), límite predictivo para una observación futura](https://itl.nist.gov/div898/software/dataplot/refman1/auxillar/predlimi.htm). [Copia consultada del 15/03/2025](https://web.archive.org/web/20250315045624/https://www.itl.nist.gov/div898/software/dataplot/refman1/auxillar/predlimi.htm).
- [11] [Caplice, C. (2006). Inventory Management: More Probabilistic Demand. MIT, ESD.260, pp. 5–6: revisión periódica](https://ocw.mit.edu/courses/esd-260j-logistics-systems-fall-2006/bc18dd1b2543535a61f826d00a8c6e42_lect12.pdf).

Los extractos factuales del emisor y los parámetros utilizados están en `data/reference/`. No se sustituyen por datos actuales al ejecutar el notebook.